# EasyVitessce Example: Modify image channel settings, then use values in Python

## Downloading and importing necessary packages

By default, interactive plots are enabled upon importing easy_vitessce. This notebook aims to demonstrate the transition between static and interactive plots, so the interactive plots are initially turned off.

In [1]:
import easy_vitessce as ev 
import spatialdata as sd
import spatialdata_plot
from os.path import join 

/Users/ericmoerth/ws/easy_vitessce/.venv/lib/python3.13/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/ericmoerth/ws/easy_vitessce/.venv/lib/python3.13/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/Users/ericmoerth/ws/easy_vitessce/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it 

## Download the data

In [2]:
import os
from os.path import join, isfile, isdir
from urllib.request import urlretrieve
import zipfile

In [3]:
data_dir = "data"
zip_path = join(data_dir, "mcmicro_io.spatialdata.zarr.zip")
sdata_path = join(data_dir, "mcmicro_io.spatialdata.zarr")

In [4]:
if not isdir(sdata_path):
    if not isfile(zip_path):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve('https://mghp.osn.xsede.org/bir190004-bucket01/BiocSpatialData/mcmicro_io.zip', zip_path)
    with zipfile.ZipFile(zip_path,"r") as zip_ref:
        zip_ref.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_path)

## Read the data

In [5]:
sdata = sd.read_zarr(sdata_path)
sdata

/Users/ericmoerth/ws/easy_vitessce/.venv/lib/python3.13/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/ericmoerth/ws/easy_vitessce/.venv/lib/python3.13/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


SpatialData object, with associated Zarr store: /Users/ericmoerth/ws/easy_vitessce/docs/notebooks/data/mcmicro_io.spatialdata.zarr
├── Images
│     └── 'exemplar-001_image': DataArray[cyx] (12, 3139, 2511)
├── Labels
│     ├── 'exemplar-001_cell': DataArray[yx] (3139, 2511)
│     └── 'exemplar-001_nuclei': DataArray[yx] (3139, 2511)
└── Tables
      ├── 'exemplar-001--ilastik_cell': AnnData (11607, 12)
      └── 'exemplar-001--unmicst_cell': AnnData (11170, 12)
with coordinate systems:
    ▸ 'global', with elements:
        exemplar-001_image (Images), exemplar-001_cell (Labels), exemplar-001_nuclei (Labels)

## Interactive plotting

Store the return value of `.pl.show()` in a variable.
See more details at https://vitessce.github.io/easy_vitessce/advanced.html#access-the-vitessce-configuration

In [6]:
vw = sdata.pl.render_images(element="exemplar-001_image").pl.show()
vw

VitessceWidget(js_dev_mode=True, uid='7bf6')

# Obtain the current channel settings

See more details at https://vitessce.github.io/easy_vitessce/advanced.html#access-values-from-the-coordination-space

In [7]:
current_config = vw._config

If you are curious about what is going on below with the "meta-coordination" stuff, see https://use-coordination.dev/ and our short paper at https://doi.org/10.1109/VIS55277.2024.00041 (especially the supplemental materials where the details are explained).

In [8]:
meta_scopes = current_config["coordinationSpace"]["metaCoordinationScopes"]
meta_scopes_by = current_config["coordinationSpace"]["metaCoordinationScopesBy"]

In [9]:
first_layer_scope = meta_scopes['init_A_image_0']['imageLayer'][0]
first_layer_scope

'init_A_image_0'

In [10]:
channel_scope_names = meta_scopes_by['init_A_image_0']['imageLayer']['imageChannel'][first_layer_scope]
channel_scope_names

['init_A_image_0',
 'init_A_image_1',
 'init_A_image_2',
 'init_A_image_3',
 'init_A_image_4']

In [11]:
channel_colors = current_config["coordinationSpace"]["spatialChannelColor"]
channel_colors

{'init_A_image_0': [0, 0, 255],
 'init_A_image_1': [0, 255, 0],
 'init_A_image_2': [255, 0, 255],
 'init_A_image_3': [255, 255, 0],
 'init_A_image_4': [0, 255, 255],
 'A': [255, 255, 255]}

In [12]:
first_channel_color = channel_colors[channel_scope_names[0]]
first_channel_color

[0, 0, 255]

In [13]:
channel_windows = current_config["coordinationSpace"]["spatialChannelWindow"]
channel_windows

{'init_A_image_0': [0, 63197],
 'init_A_image_1': [0, 25683],
 'init_A_image_2': [0, 65535],
 'init_A_image_3': [0, 29144],
 'init_A_image_4': [0, 47994],
 'A': None}

In [14]:
first_channel_window = channel_windows[channel_scope_names[0]]
first_channel_window

[0, 63197]